In [1]:
import fastf1
import pandas as pd
import numpy as np
import requests
import time

/Users/davidle/Documents/f1-strategy-predictor/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
fastf1.Cache.enable_cache('../data/raw')

def get_season_results(year):
    rows = []
    offset = 0
    limit = 100

    while True:
        url = f"https://api.jolpi.ca/ergast/f1/{year}/results.json?limit={limit}&offset={offset}"
        response = requests.get(url)
        data = response.json()
        total = int(data['MRData']['total'])
        races = data['MRData']['RaceTable']['Races']
        
        
        for race in races:
            circuit   = race['Circuit']['circuitId']
            round_num = int(race['round'])
            
            for result in race['Results']:
                rows.append({
                    'year':     year,
                    'round':    round_num,
                    'circuit':  circuit,
                    'driver':   result['Driver']['driverId'],
                    'team':     result['Constructor']['constructorId'],
                    'grid':     int(result['grid']),
                    'position': int(result['position']),
                    'won':      1 if result['position'] == '1' else 0,
                    'points':   float(result['points']),
                    'laps':     int(result['laps']),
                    'status':   result['status'],
                })
        
        offset += limit
        
        if offset >= total:
            break
            
        time.sleep(1)
    return pd.DataFrame(rows)

all_results = []
for year in range(2015, 2027):
    print(f"Pulling {year}...")
    df = get_season_results(year)
    all_results.append(df)
    time.sleep(2)

results_df = pd.concat(all_results, ignore_index=True)
results_df.to_csv('../data/raw/historical_results.csv', index=False)
results_df.head(10)

Pulling 2015...
Pulling 2016...
Pulling 2017...
Pulling 2018...
Pulling 2019...
Pulling 2020...
Pulling 2021...
Pulling 2022...
Pulling 2023...
Pulling 2024...
Pulling 2025...
Pulling 2026...


,year,round,circuit,driver,team,grid,position,won,points,laps,status
0,2015,1,albert_park,hamilton,mercedes,1,1,1,25.0,58,Finished
1,2015,1,albert_park,rosberg,mercedes,2,2,0,18.0,58,Finished
2,2015,1,albert_park,vettel,ferrari,4,3,0,15.0,58,Finished
3,2015,1,albert_park,massa,williams,3,4,0,12.0,58,Finished
4,2015,1,albert_park,nasr,sauber,10,5,0,10.0,58,Finished
5,2015,1,albert_park,ricciardo,red_bull,6,6,0,8.0,57,+1 Lap
6,2015,1,albert_park,hulkenberg,force_india,13,7,0,6.0,57,+1 Lap
7,2015,1,albert_park,ericsson,sauber,15,8,0,4.0,57,+1 Lap
8,2015,1,albert_park,sainz,toro_rosso,7,9,0,2.0,57,+1 Lap
9,2015,1,albert_park,perez,force_india,14,10,0,1.0,57,+1 Lap


In [3]:
def get_qualifiying_results(year):
    rows   = []
    limit  = 100
    offset = 0

    while True:
        url = f"https://api.jolpi.ca/ergast/f1/{year}/qualifying.json?limit={limit}&offset={offset}"
        
        try:
            response = requests.get(url, timeout=10)
            
            # Empty response — rate limited or server error
            if not response.text.strip():
                print(f"  Empty response for {year} offset {offset}, waiting 5s...")
                time.sleep(5)
                continue
            
            data  = response.json()
            total = int(data['MRData']['total'])
            races = data['MRData']['RaceTable']['Races']

            for race in races:
                circuit   = race['Circuit']['circuitId']
                round_num = int(race['round'])
                for result in race['QualifyingResults']:
                    rows.append({
                        'year':      year,
                        'round':     round_num,
                        'circuit':   circuit,
                        'driver':    result['Driver']['driverId'],
                        'quali_pos': int(result['position'])
                    })

            offset += limit
            if offset >= total:
                break

            time.sleep(0.5)

        except requests.exceptions.JSONDecodeError:
            print(f"  Bad response for {year} offset {offset}, retrying in 5s...")
            time.sleep(5)
            continue
        except Exception as e:
            print(f"  Error for {year} offset {offset}: {e}, skipping...")
            break

    return pd.DataFrame(rows)
quali_results=[]
for year in range(2015,2027):
    print(f"pulling quali {year}...")
    df=get_qualifiying_results(year)
    quali_results.append(df)

quali_df=pd.concat(quali_results,ignore_index=True)
quali_df.to_csv('../data/raw/qualifying_results.csv', index=False)
quali_df.head(10)

    

pulling quali 2015...
pulling quali 2016...
pulling quali 2017...
pulling quali 2018...
pulling quali 2019...
pulling quali 2020...
pulling quali 2021...
pulling quali 2022...
pulling quali 2023...
pulling quali 2024...
pulling quali 2025...
pulling quali 2026...


,year,round,circuit,driver,quali_pos
0,2015,1,albert_park,hamilton,1
1,2015,1,albert_park,rosberg,2
2,2015,1,albert_park,massa,3
3,2015,1,albert_park,vettel,4
4,2015,1,albert_park,raikkonen,5
5,2015,1,albert_park,bottas,6
6,2015,1,albert_park,ricciardo,7
7,2015,1,albert_park,sainz,8
8,2015,1,albert_park,grosjean,9
9,2015,1,albert_park,maldonado,10


In [4]:
#get qualifiying gap 
#driver mapping 
def parse_laptime(t):
    if not t or t == 'N/A':
        return None
    try:
        if ':' in t:
            parts   = t.split(':')
            minutes = int(parts[0])
            seconds = float(parts[1])
            return minutes * 60 + seconds
        else:
            return float(t)
    except:
        return None

def get_quali_gaps(year, round_num, retries=3):
    url = f"https://api.jolpi.ca/ergast/f1/{year}/{round_num}/qualifying.json"
    
    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=10)
            if not response.text.strip():
                print(f"  {year} R{round_num}: empty, retrying ({attempt+1}/{retries})...")
                time.sleep(3)
                continue

            data  = response.json()
            races = data['MRData']['RaceTable']['Races']
            if not races:
                print(f"  {year} R{round_num}: no races ❌")
                return None

            rows = []
            for result in races[0]['QualifyingResults']:
                driver_id   = result['Driver']['driverId']
                times       = [
                    parse_laptime(result.get('Q1')),
                    parse_laptime(result.get('Q2')),
                    parse_laptime(result.get('Q3')),
                ]
                valid_times = [t for t in times if t is not None]
                if not valid_times:
                    continue
                rows.append({
                    'driver':    driver_id,
                    'best_time': min(valid_times),
                    'year':      year,
                    'round':     round_num,
                })

            if not rows:
                print(f"  {year} R{round_num}: no valid times ❌")
                return None

            df             = pd.DataFrame(rows)
            pole_time      = df['best_time'].min()
            df['quali_gap'] = df['best_time'] - pole_time
            print(f"  {year} R{round_num}: ✅ ({len(df)} drivers)")
            return df[['year', 'round', 'driver', 'quali_gap']]

        except Exception as e:
            print(f"  {year} R{round_num}: error — {e}, retrying ({attempt+1}/{retries})...")
            time.sleep(3)

    print(f"  {year} R{round_num}: failed after {retries} attempts ❌")
    return None

ROUNDS_PER_YEAR = {
    2015: range(1, 20),
    2016: range(1, 22),
    2017: range(1, 21),
    2018: range(1, 22),
    2019: range(1, 22),
    2020: range(1, 18),
    2021: range(1, 23),
    2022: range(1, 23),
    2023: range(1, 23),
    2024: range(1, 25),
    2025: range(1, 25),
    2026: range(1, 3),
}

# ── Load existing data if file exists ─────────────────────
import os

if os.path.exists('../data/processed/quali_gaps.csv'):
    existing_gaps  = pd.read_csv('../data/processed/quali_gaps.csv')
    already_pulled = set(zip(existing_gaps['year'], existing_gaps['round']))
    all_gaps       = [existing_gaps]
    print(f"Loaded existing: {len(existing_gaps)} rows across years {sorted(existing_gaps['year'].unique())}")
else:
    existing_gaps  = pd.DataFrame()
    already_pulled = set()
    all_gaps       = []
    print("No existing file — pulling everything from scratch")

# ── Only pull missing rounds ───────────────────────────────
for year in range(2015, 2027):
    year_missing = [r for r in ROUNDS_PER_YEAR[year] if (year, r) not in already_pulled]
    
    if not year_missing:
        print(f"{year}: already complete ✅")
        continue
    
    print(f"\nPulling {year} — {len(year_missing)} rounds missing...")
    
    for round_num in year_missing:
        gaps = get_quali_gaps(year, round_num)
        if gaps is not None:
            all_gaps.append(gaps)
            # Save after every successful pull — never lose progress
            pd.concat(all_gaps, ignore_index=True) \
              .drop_duplicates(subset=['year', 'round', 'driver']) \
              .to_csv('../data/processed/quali_gaps.csv', index=False)
        time.sleep(2)

# ── Final save ────────────────────────────────────────────
gaps_df = pd.concat(all_gaps, ignore_index=True) \
            .drop_duplicates(subset=['year', 'round', 'driver']) \
            .sort_values(['year', 'round', 'driver']) \
            .reset_index(drop=True)

gaps_df.to_csv('../data/processed/quali_gaps.csv', index=False)
print(f"\n✅ Total rows: {len(gaps_df)}")
print(f"Years: {sorted(gaps_df['year'].unique())}")
print(gaps_df.head(10))



Loaded existing: 4676 rows across years [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
2015: already complete ✅
2016: already complete ✅
2017: already complete ✅
2018: already complete ✅
2019: already complete ✅
2020: already complete ✅
2021: already complete ✅
2022: already complete ✅
2023: already complete ✅
2024: already complete ✅
2025: already complete ✅
2026: already complete ✅

✅ Total rows: 4676
Years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
   year  round           driver  quali_gap
0  2015      1           bottas      1.469
1  2015      1           button      5.095
2  2015      1         ericsson      5.049
3  2015      1         grosjean      2.233
4  2015      1         hamilton 

In [5]:
def get_constructor_standings(year, round_num, retries=3):
    prev_round = max(1, round_num - 1)
    url        = f"https://api.jolpi.ca/ergast/f1/{year}/{prev_round}/constructorStandings.json"

    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=10)
            if not response.text.strip():
                print(f"  {year} R{round_num}: empty, retrying ({attempt+1}/{retries})...")
                time.sleep(3)
                continue

            data            = response.json()
            standings_table = data['MRData']['StandingsTable']['StandingsLists']
            if not standings_table:
                return None

            rows        = []
            standings   = standings_table[0]['ConstructorStandings']
            total_teams = len(standings)

            for i, s in enumerate(standings):
                position = int(s.get('position', i + 1))
                rows.append({
                    'year':                  year,
                    'round':                 round_num,
                    'team':                  s['Constructor']['constructorId'],
                    'constructor_rank_norm': 1 - (position - 1) / max(total_teams - 1, 1),
                })

            if not rows:
                return None

            return pd.DataFrame(rows)

        except Exception as e:
            print(f"  {year} R{round_num}: attempt {attempt+1} failed — {e}")
            time.sleep(3)

    print(f"  {year} R{round_num}: failed after {retries} attempts ❌")
    return None

# ── Load existing data if file exists ─────────────────────
import os

if os.path.exists('../data/processed/constructor_standings.csv'):
    existing_standings = pd.read_csv('../data/processed/constructor_standings.csv')
    already_pulled     = set(zip(existing_standings['year'], existing_standings['round']))
    all_standings      = [existing_standings]
    print(f"Loaded existing: {len(existing_standings)} rows across years {sorted(existing_standings['year'].unique())}")
else:
    existing_standings = pd.DataFrame()
    already_pulled     = set()
    all_standings      = []
    print("No existing file — pulling everything from scratch")

# ── Only pull missing rounds ───────────────────────────────
for year in range(2015, 2027):
    year_missing = [r for r in ROUNDS_PER_YEAR[year] if (year, r) not in already_pulled]

    if not year_missing:
        print(f"{year}: already complete ✅")
        continue

    print(f"\nPulling {year} constructor standings — {len(year_missing)} rounds missing...")

    for round_num in year_missing:
        s = get_constructor_standings(year, round_num)
        if s is not None:
            all_standings.append(s)
            # Save after every successful pull
            pd.concat(all_standings, ignore_index=True) \
              .drop_duplicates(subset=['year', 'round', 'team']) \
              .to_csv('../data/processed/constructor_standings.csv', index=False)
        time.sleep(3)

# ── Final save ────────────────────────────────────────────
standings_df = pd.concat(all_standings, ignore_index=True) \
                 .drop_duplicates(subset=['year', 'round', 'team']) \
                 .sort_values(['year', 'round', 'team']) \
                 .reset_index(drop=True)

standings_df.to_csv('../data/processed/constructor_standings.csv', index=False)
print(f"\n✅ Total rows: {len(standings_df)}")
print(f"Years: {sorted(standings_df['year'].unique())}")
print(standings_df.head(10))

Loaded existing: 2371 rows across years [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
2015: already complete ✅
2016: already complete ✅
2017: already complete ✅
2018: already complete ✅
2019: already complete ✅
2020: already complete ✅
2021: already complete ✅
2022: already complete ✅
2023: already complete ✅
2024: already complete ✅
2025: already complete ✅
2026: already complete ✅

✅ Total rows: 2371
Years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
   year  round         team  constructor_rank_norm
0  2015      1      ferrari                  0.875
1  2015      1  force_india                  0.375
2  2015      1     lotus_f1                  0.000
3  2015      1      mclaren                

In [6]:
import time
import os

all_tyres   = []
all_weather = []

if os.path.exists('../data/processed/tyre_data.csv'):
    existing = pd.read_csv('../data/processed/tyre_data.csv')
    already_pulled = set(zip(existing['year'], existing['round']))
    all_tyres.append(existing)
    print(f"Already have tyre data for {len(already_pulled)} rounds")
else:
    already_pulled = set()

if os.path.exists('../data/processed/race_weather.csv'):
    existing_w = pd.read_csv('../data/processed/race_weather.csv')
    already_pulled_w = set(zip(existing_w['year'], existing_w['round']))
    all_weather.append(existing_w)
    print(f"Already have weather data for {len(already_pulled_w)} rounds")
else:
    already_pulled_w = set()

for year in range(2018, 2027):
    print(f"\nPulling {year}...")
    for round_num in ROUNDS_PER_YEAR[year]:

        need_tyre    = (year, round_num) not in already_pulled
        need_weather = (year, round_num) not in already_pulled_w

        if not need_tyre and not need_weather:
            print(f"  {year} R{round_num}: skipping — already have both")
            continue

        try:
            session = fastf1.get_session(year, round_num, 'R')
            session.load(telemetry  = False,  
                        weather    = True,   
                        messages   = False,  
                        laps       = True,)

            # Tyres
            if need_tyre:
                first_laps = session.laps[session.laps['LapNumber'] == 1]
                rows = []
                for _, lap in first_laps.iterrows():
                    compound = lap.get('Compound')
                    driver_num = lap.get('Driver')
                    if not driver_num or not compound:
                        continue
                    try:
                        driver_info = session.get_driver(driver_num)
                        driver_id   = driver_info['DriverId']
                    except:
                        driver_id = DRIVER_MAP.get(driver_num)
                    if not driver_id:
                        continue
                    rows.append({
                        'year':              year,
                        'round':             round_num,
                        'driver':            driver_id,
                        'starting_compound': {
                            'SOFT': 0, 'MEDIUM': 1, 'HARD': 2,
                            'INTER': 3, 'WET': 4
                        }.get(str(compound).upper(), 1),
                    })
                if rows:
                    all_tyres.append(pd.DataFrame(rows))
                    print(f"  {year} R{round_num}: tyres ✅ ({len(rows)} drivers)")

            # Weather
            if need_weather and session.weather_data is not None and not session.weather_data.empty:
                weather = session.weather_data
                all_weather.append(pd.DataFrame([{
                    'year':       year,
                    'round':      round_num,
                    'wet_race':   int(weather['Rainfall'].any()),
                    'track_temp': round(float(weather['TrackTemp'].mean()), 1),
                    'air_temp':   round(float(weather['AirTemp'].mean()), 1),
                }]))
                print(f"  {year} R{round_num}: weather ✅")

            # Save after every successful session — resume safe
            if all_tyres:
                pd.concat(all_tyres, ignore_index=True).drop_duplicates(
                    subset=['year', 'round', 'driver']
                ).to_csv('../data/processed/tyre_data.csv', index=False)

            if all_weather:
                pd.concat(all_weather, ignore_index=True).drop_duplicates(
                    subset=['year', 'round']
                ).to_csv('../data/processed/race_weather.csv', index=False)

        except Exception as e:
            print(f"  {year} R{round_num}: failed — {e}")

        time.sleep(8)  # 8 seconds between sessions = ~450 sessions/hour under the limit

print("\n✅ Done")


Already have tyre data for 175 rounds
Already have weather data for 175 rounds

Pulling 2018...
  2018 R1: skipping — already have both
  2018 R2: skipping — already have both
  2018 R3: skipping — already have both
  2018 R4: skipping — already have both
  2018 R5: skipping — already have both
  2018 R6: skipping — already have both
  2018 R7: skipping — already have both
  2018 R8: skipping — already have both
  2018 R9: skipping — already have both
  2018 R10: skipping — already have both
  2018 R11: skipping — already have both
  2018 R12: skipping — already have both
  2018 R13: skipping — already have both
  2018 R14: skipping — already have both
  2018 R15: skipping — already have both
  2018 R16: skipping — already have both
  2018 R17: skipping — already have both
  2018 R18: skipping — already have both
  2018 R19: skipping — already have both
  2018 R20: skipping — already have both
  2018 R21: skipping — already have both

Pulling 2019...
  2019 R1: skipping — already have 

In [7]:
import pandas as pd
tyres   = pd.read_csv('../data/processed/tyre_data.csv')
weather = pd.read_csv('../data/processed/race_weather.csv')

print(f"Tyres:   {len(tyres['year'].unique())} years, {len(tyres)} rows")
print(f"Weather: {len(weather['year'].unique())} years, {len(weather)} rows")
print(f"Tyre years:   {sorted(tyres['year'].unique())}")
print(f"Weather years:{sorted(weather['year'].unique())}")

Tyres:   9 years, 3495 rows
Weather: 9 years, 175 rows
Tyre years:   [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Weather years:[np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


In [9]:
df = results_df.merge(quali_df[['year','round','driver','quali_pos']],
                    on=['year','round','driver'],
                    how='left')

df = df.merge(gaps_df[['year', 'round', 'driver', 'quali_gap']],
    on=['year', 'round', 'driver'],
    how='left'
)
df['quali_gap'] = df['quali_gap'].fillna(df['quali_gap'].median())
print(f"After gaps merge: {len(df)} rows")
print(f"Nulls in quali_gap: {df['quali_gap'].isnull().sum()}")
print(f"Average quali gap:  {df['quali_gap'].mean():.3f}s")

# ── Merge constructor standings ───────────────────────────
try:
    standings_df = pd.read_csv('../data/processed/constructor_standings.csv')
    df = df.merge(
        standings_df[['year', 'round', 'team', 'constructor_rank_norm']],
        on=['year', 'round', 'team'],
        how='left'
    )
    df['constructor_rank_norm'] = df['constructor_rank_norm'].fillna(0.5)
    print(f"✅ Constructor standings merged")
except FileNotFoundError:
    df['constructor_rank_norm'] = 0.5
    print("⚠️  No constructor standings found — defaulting to 0.5")

# ── Merge tyre data ───────────────────────────────────────
try:
    tyres_df = pd.read_csv('../data/processed/tyre_data.csv')
    df = df.merge(
        tyres_df[['year', 'round', 'driver', 'starting_compound']],
        on=['year', 'round', 'driver'],
        how='left'
    )
    df['starting_compound'] = df['starting_compound'].fillna(1)
    print(f"✅ Tyre data merged")
except FileNotFoundError:
    df['starting_compound'] = 1
    print("⚠️  No tyre data found — defaulting to medium")

# ── Merge weather data ────────────────────────────────────
try:
    weather_df = pd.read_csv('../data/processed/race_weather.csv')
    df = df.merge(
        weather_df[['year', 'round', 'wet_race', 'track_temp']],
        on=['year', 'round'],
        how='left'
    )
    df['wet_race']   = df['wet_race'].fillna(0).astype(int)
    df['track_temp'] = df['track_temp'].fillna(df['track_temp'].median())
    print(f"✅ Weather merged — {int(df['wet_race'].sum() / 20)} wet races found")
except FileNotFoundError:
    df['wet_race']   = 0
    df['track_temp'] = 35.0
    print("⚠️  No weather data found — defaulting to dry")

# ── Circuit type ──────────────────────────────────────────
CIRCUIT_TYPES = {
    'monaco':        0, 'baku':          0, 'marina_bay':  0,
    'jeddah':        0, 'vegas':         0, 'albert_park': 0,
    'villeneuve':    0, 'ifema_madrid':  0,
    'monza':         1, 'spa':           1, 'silverstone': 1,
    'red_bull_ring': 1, 'zandvoort':     1, 'suzuka':      1,
    'bahrain':       2, 'shanghai':      2, 'catalunya':   2,
    'hungaroring':   2, 'americas':      2, 'rodriguez':   2,
    'interlagos':    2, 'yas_marina':    2, 'losail':      2,
    'imola':         2, 'miami':         2,
}
df['circuit_type'] = df['circuit'].map(CIRCUIT_TYPES).fillna(2)
print(f"✅ Circuit types mapped")

win_rate_by_grid = df.groupby('grid')['won'].mean()
print(win_rate_by_grid.head(5))

# ── Feature engineering ───────────────────────────────────
df = df.sort_values(['driver','year','round'])

df['driver_form'] = (
    df.groupby('driver')['position']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    .fillna(10.5)  # midfield default for new drivers
)

team_wins = (
    df.groupby(['year', 'team'])['won']
    .transform(lambda x: x.shift(1).expanding().mean())
    .fillna(0)
)
df['team_win_rate'] = team_wins

circuit_wins = (
    df.groupby(['driver', 'circuit'])['won']
    .transform(lambda x: x.shift(1).expanding().sum())
    .fillna(0)
)
df['circuit_wins_before'] = circuit_wins

df['finished'] = (~df['status'].str.contains(
    'Lap|Accident|Collision|Engine|Gearbox', na=False)).astype(int)
df['driver_finish_rate'] = (
    df.groupby('driver')['finished']
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
    .fillna(0.9)
)

df['grid_penalty'] = (df['grid'] - df['quali_pos']).clip(lower=0).fillna(0)

# ── Circuit average finish ────────────────────────────────
# How does this driver typically finish at this specific circuit
# vs their overall average — captures circuit specialists
df['circuit_avg_finish'] = (
    df.groupby(['driver', 'circuit'])['position']
    .transform(lambda x: x.shift(1).expanding().mean())
    .fillna(10.5)  # midfield default if no history
)

# How much better/worse does this driver do at this circuit
# vs their recent overall form
# Negative = overperforms here, Positive = underperforms here
df['circuit_overperformance'] = (
    df['circuit_avg_finish'] - df['driver_form']
)

# ── Constructor momentum ──────────────────────────────────
# Is this team improving or declining in the standings
# over the last 3 rounds
df['constructor_momentum'] = (
    df.groupby(['year', 'team'])['constructor_rank_norm']
    .transform(lambda x: x.diff(3))
    .fillna(0)
)
# Positive = team moved up in standings over last 3 rounds
# Negative = team dropped in standings
# Zero = stable or no history

print(df[['driver', 'grid', 'quali_pos', 'quali_gap', 'driver_form',
          'team_win_rate', 'circuit_wins_before', 'won']].head(20))


After gaps merge: 4742 rows
Nulls in quali_gap: 0
Average quali gap:  1.776s
✅ Constructor standings merged
✅ Tyre data merged
✅ Weather merged — 44 wet races found
✅ Circuit types mapped
grid
0    0.000000
1    0.544681
2    0.235043
3    0.106383
4    0.038298
Name: won, dtype: float64
      driver  grid  quali_pos  quali_gap  driver_form  team_win_rate  \
2395  aitken    17       18.0      1.515         10.5            0.0   
1673   albon    13       13.0      2.150         10.5            0.0   
1688   albon    12       12.0      1.647         14.0            0.0   
1709   albon     0        NaN      1.407         11.5            0.0   
1730   albon    11       12.0      1.659         11.0            0.0   
1750   albon    11       12.0      2.039         11.0            0.0   
1767   albon    10       10.0      1.263         11.0            0.0   
1798   albon    13       14.0      1.780          9.8            0.0   
1814   albon    11       11.0      2.142         11.8          

In [12]:
from sklearn.preprocessing import LabelEncoder
team_encoder=LabelEncoder()
circuit_encoder=LabelEncoder()
df['team_encoded']=team_encoder.fit_transform(df['team'])
df['circuit_encoded']=circuit_encoder.fit_transform(df['circuit'])

FEATURES = [
    'grid',
    'quali_pos',
    'quali_gap',
    'grid_penalty',
    'driver_form',
    'team_win_rate',
    'circuit_wins_before',
    'circuit_avg_finish',
    'circuit_overperformance',
    'driver_finish_rate',
    'constructor_rank_norm',
    'constructor_momentum',
    'starting_compound',
    'circuit_type',
    'wet_race',             
    'track_temp',          
    'team_encoded',
    'circuit_encoded',
]
TARGET='position'

df_clean = df.dropna(subset=FEATURES)
print(f"Clean rows: {len(df_clean)}")
print(f"Features:   {len(FEATURES)}")
print(f"Years:      {sorted(df_clean['year'].unique())}")


Clean rows: 4724
Features:   18
Years:      [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


In [13]:
train_df=df_clean[df_clean['year']<=2024]
test_df=df_clean[df_clean['year']==2025]

X_train=train_df[FEATURES]
y_train=train_df[TARGET]

X_test=test_df[FEATURES]
y_test=test_df[TARGET]

print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")
print(f"Train position range: {y_train.min()} to {y_train.max()}")
print(f"Test position range:  {y_test.min()} to {y_test.max()}")
print(f"Average predicted position: {y_train.mean():.1f}")


Train: 4205 rows | Test: 478 rows
Train position range: 1 to 22
Test position range:  1 to 20
Average predicted position: 10.6


In [14]:
from sklearn.linear_model import Ridge
from sklearn.metrics import ndcg_score, mean_absolute_error 
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
import matplotlib.pyplot as plt


scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.fit_transform(X_test)

#ridge instead of Linear regression
baseline=Ridge()
baseline.fit(X_train_scaled,y_train)

baseline_preds=baseline.predict(X_test_scaled)

#metrics
#spearman
correlation, _ =spearmanr(baseline_preds,y_test)
print(f"Baseline Spearman:{correlation:.3f}")

#ncdg
actual_relevance=(21-y_test.values).reshape(1,-1)
predicted_relevance=(21-baseline_preds).reshape(1,-1)
ndcg=ndcg_score(actual_relevance,predicted_relevance)
print(f"Basline NDCG: {ndcg:.3f}")

test_check=test_df.copy()

test_check['predicted_pos']=baseline_preds

#winner accuracy 
correct=0
total=test_check['round'].nunique()
for round_num in test_check['round'].unique():
    race=test_check[test_check['round']==round_num]
    predicted_winner=race.loc[race['predicted_pos'].idxmin(),'driver']
    actual_winner=race.loc[race['position']==1,'driver'].values[0]
    if predicted_winner==actual_winner:
        correct+=1
print(f"winner accuracy: {correct}/{total} ({correct/total:.1%})")

#podium accuracy
correct_podium=0
for round_num in test_check['round'].unique():
    race= test_check[test_check['round']==round_num]
    predicted_top3=set(race.nsmallest(3,'predicted_pos')['driver'])
    actual_top3=set(race.nsmallest(3,'position')['driver'])
    if predicted_top3==actual_top3:
        correct_podium+=1
print(f"Podium accuracy: {correct_podium}/{total} ({correct_podium/total:.1%})")

mae=mean_absolute_error(y_test,baseline_preds)
print(f"Baseline Mae:{mae:.2f} positions off on average")

print(f"\nTolerance accuracy:")
for tolerance in [1,2,3,5]:
    correct=(abs(baseline_preds-y_test)<=tolerance).mean()
    print(f"  Within {tolerance} position(s): {correct:.1%}")





Baseline Spearman:0.656
Basline NDCG: 0.973
winner accuracy: 10/24 (41.7%)
Podium accuracy: 8/24 (33.3%)
Baseline Mae:3.27 positions off on average

Tolerance accuracy:
  Within 1 position(s): 18.8%
  Within 2 position(s): 37.0%
  Within 3 position(s): 55.4%
  Within 5 position(s): 82.6%


In [ ]:
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

# Check predicted probabilities distribution
baseline_preds = baseline.predict(X_test_scaled)

plt.hist(baseline_preds, bins=50)
plt.title('Predicted Win Probabilities Distribution')
plt.xlabel('Predicted Probability')
plt.show()

# Check if model is just predicting everything as 0
print(f"Max probability:  {baseline_preds.max():.3f}")
print(f"Min probability:  {baseline_preds.min():.3f}")
print(f"Mean probability: {baseline_preds.mean():.3f}")

# Most importantly — check top predictions vs actual winners
test_df_check = test_df.copy()
test_df_check['predicted_pos'] = baseline_preds

# For each race did the highest probability driver actually win?
correct = 0
total_races = test_df_check['round'].nunique()

for round_num in test_df_check['round'].unique():
    race = test_df_check[test_df_check['round'] == round_num]
    predicted_winner = race.loc[race['predicted_pos'].idxmin(), 'driver']
    actual_winner    = race.loc[race['won'] == 1, 'driver'].values[0]
    if predicted_winner == actual_winner:
        correct += 1

print(f"Correctly predicted winner: {correct}/{total_races} races")
print(f"Win prediction accuracy: {correct/total_races:.1%}")


In [15]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import make_scorer
from scipy.stats import spearmanr
import xgboost as xgb
import numpy as np
import itertools

# Use all data up to 2025 for tuning
# Keep 2026 completely out — it's your deployment year
tune_df = df_clean[df_clean['year'] <= 2025]
X_tune  = tune_df[FEATURES]
y_tune  = tune_df[TARGET]

# Custom Spearman scorer
def spearman_scorer(y_true, y_pred):
    corr, _ = spearmanr(y_pred, y_true)
    return corr

scorer = make_scorer(spearman_scorer)

# Parameter grid — wider ranges now that we have more data
param_grid = {
    'n_estimators':     [100, 200, 300],
    'max_depth':        [3, 4, 5],
    'learning_rate':    [0.01, 0.05, 0.1],
    'subsample':        [0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'min_child_weight': [1, 3, 5],
    'reg_lambda':       [1, 2, 5],
}

# Time-based cross validation — always train on past, test on future
# This is more honest than random KFold for time series data
test_years  = [2021, 2022, 2023, 2024, 2025]
best_score  = -999
best_params = None
results_log = []

# Start with current best params as baseline
baseline_params = {
    'n_estimators':     100,
    'max_depth':        3,
    'learning_rate':    0.05,
    'subsample':        0.9,
    'colsample_bytree': 0.7,
    'min_child_weight': 1,
    'reg_lambda':       1,
}

def evaluate_params(params, test_years=test_years):
    scores = []
    for test_year in test_years:
        train = df_clean[df_clean['year'] < test_year]
        test  = df_clean[df_clean['year'] == test_year]
        train = train.dropna(subset=FEATURES)
        test  = test.dropna(subset=FEATURES)
        if test.empty or len(train) < 100:
            continue
        model = xgb.XGBRegressor(
            **params,
            random_state=42,
            eval_metric='mae',
            verbosity=0,
        )
        model.fit(train[FEATURES], train[TARGET])
        preds       = model.predict(test[FEATURES])
        corr, _     = spearmanr(preds, test[TARGET])
        scores.append(corr)
    return np.mean(scores) if scores else -999

# Evaluate baseline first
baseline_score = evaluate_params(baseline_params)
print(f"Baseline score: {baseline_score:.4f}")
print(f"Params: {baseline_params}")
print()

# Grid search — test key combinations
# Focus on parameters that matter most
search_space = [
    {'n_estimators': n, 'max_depth': d, 'learning_rate': lr,
     'subsample': ss, 'colsample_bytree': cbt,
     'min_child_weight': mcw, 'reg_lambda': rl}
    for n   in [100, 200, 300]
    for d   in [3, 4, 5]
    for lr  in [0.01, 0.05, 0.1]
    for ss  in [0.8, 0.9]
    for cbt in [0.7, 0.8]
    for mcw in [1, 3]
    for rl  in [1, 2]
]

print(f"Testing {len(search_space)} parameter combinations...")
print(f"This will take a few minutes...\n")

for i, params in enumerate(search_space):
    score = evaluate_params(params)
    results_log.append({**params, 'score': score})
    if score > best_score:
        best_score  = score
        best_params = params.copy()
        print(f"New best at combination {i+1}: {score:.4f} → {params}")

print(f"\n{'═' * 60}")
print(f"BEST PARAMS:  {best_params}")
print(f"BEST SCORE:   {best_score:.4f}")
print(f"BASELINE:     {baseline_score:.4f}")
print(f"IMPROVEMENT:  +{best_score - baseline_score:.4f}")
print(f"{'═' * 60}")

# Show top 5 results
results_df_tune = pd.DataFrame(results_log).sort_values('score', ascending=False)
print(f"\nTop 5 parameter sets:")
print(results_df_tune.head())

Baseline score: 0.6697
Params: {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.05, 'subsample': 0.9, 'colsample_bytree': 0.7, 'min_child_weight': 1, 'reg_lambda': 1}

Testing 432 parameter combinations...
This will take a few minutes...

New best at combination 1: 0.6754 → {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.01, 'subsample': 0.8, 'colsample_bytree': 0.7, 'min_child_weight': 1, 'reg_lambda': 1}
New best at combination 145: 0.6770 → {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.01, 'subsample': 0.8, 'colsample_bytree': 0.7, 'min_child_weight': 1, 'reg_lambda': 1}

════════════════════════════════════════════════════════════
BEST PARAMS:  {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.01, 'subsample': 0.8, 'colsample_bytree': 0.7, 'min_child_weight': 1, 'reg_lambda': 1}
BEST SCORE:   0.6770
BASELINE:     0.6697
IMPROVEMENT:  +0.0074
════════════════════════════════════════════════════════════

Top 5 parameter sets:
     n_estimators  max_

In [17]:
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV


neg=(y_train==0).sum()
pos=(y_train==1).sum()
scale=neg/pos
print(f"scale_pos_weight:{scale:.1f}")

xgb_model=xgb.XGBRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=.01,
    subsample=.8,
    colsample_bytree=.7,
    min_child_weight=1,
    reg_lambda=1,
    random_state=42,
    eval_metric="mae"
)

xgb_model.fit(
    X_train, y_train, 
    eval_set=[(X_test,y_test)],
    verbose=False
)

xgb_preds=xgb_model.predict(X_test)


scale_pos_weight:0.0


In [18]:
#spearman
xgb_spearman, _ = spearmanr(xgb_preds,y_test)
print(f"XGBoost Spearman:{xgb_spearman:.3f}")

#ndcg
xgb_ndcg=ndcg_score(
    (21-y_test.values).reshape(1,-1),
    (21-xgb_preds).reshape(1,-1)
    )
print(f"XGBoost NDCG:{xgb_ndcg:.3f}")

#mae
xgb_mae=mean_absolute_error(y_test,xgb_preds)
print(f"XGBoost MAE: {xgb_mae:.2f} positions off")

#mae tolerance
print(f"\nTolerance accuracy:")
for tolerance in [1, 2, 3, 5]:
    correct = (abs(xgb_preds - y_test) <= tolerance).mean()
    print(f"  Within {tolerance} position(s): {correct:.1%}")

# winner accuracy
test_check_xgb               = test_df.copy()
test_check_xgb['predicted_pos'] = xgb_preds

correct = 0
total   = test_check_xgb['round'].nunique()

for round_num in test_check_xgb['round'].unique():
    race             = test_check_xgb[test_check_xgb['round'] == round_num]
    predicted_winner = race.loc[race['predicted_pos'].idxmin(), 'driver']
    actual_winner    = race.loc[race['won'] == 1, 'driver'].values[0]
    if predicted_winner == actual_winner:
        correct += 1

print(f"\nWinner accuracy: {correct}/{total} ({correct/total:.1%})")

# Podium accuracy
correct_podium = 0
for round_num in test_check_xgb['round'].unique():
    race           = test_check_xgb[test_check_xgb['round'] == round_num]
    predicted_top3 = set(race.nsmallest(3, 'predicted_pos')['driver'])
    actual_top3    = set(race.nsmallest(3, 'position')['driver'])
    if predicted_top3 == actual_top3:
        correct_podium += 1

print(f"Podium accuracy: {correct_podium}/{total} ({correct_podium/total:.1%})")

XGBoost Spearman:0.672
XGBoost NDCG:0.970
XGBoost MAE: 3.34 positions off

Tolerance accuracy:
  Within 1 position(s): 17.8%
  Within 2 position(s): 35.1%
  Within 3 position(s): 50.8%
  Within 5 position(s): 81.0%

Winner accuracy: 11/24 (45.8%)
Podium accuracy: 7/24 (29.2%)


In [19]:
for round_num in test_check_xgb['round'].unique():
    race             = test_check_xgb[test_check_xgb['round'] == round_num]
    predicted_winner = race.loc[race['predicted_pos'].idxmin(), 'driver']
    actual_winner    = race.loc[race['won'] == 1, 'driver'].values[0]
    pred_pos         = race['predicted_pos'].min()
    status           = "✅" if predicted_winner == actual_winner else "❌"
    print(f"{status} Round {round_num:2d}: predicted {predicted_winner:<25} actual {actual_winner:<25} min_pred={pred_pos:.2f}")

❌ Round  1: predicted piastri                   actual norris                    min_pred=5.26
❌ Round  2: predicted norris                    actual piastri                   min_pred=4.51
❌ Round  3: predicted norris                    actual max_verstappen            min_pred=4.22
✅ Round  4: predicted piastri                   actual piastri                   min_pred=4.61
✅ Round  5: predicted piastri                   actual piastri                   min_pred=4.58
❌ Round  6: predicted norris                    actual piastri                   min_pred=4.50
❌ Round  7: predicted piastri                   actual max_verstappen            min_pred=4.39
✅ Round  8: predicted norris                    actual norris                    min_pred=4.41
✅ Round  9: predicted piastri                   actual piastri                   min_pred=4.61
❌ Round 10: predicted piastri                   actual russell                   min_pred=4.83
✅ Round 11: predicted norris                    ac

In [20]:
import joblib
import os

# Create models folder if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save everything
joblib.dump(xgb_model,       '../models/position_ranker.pkl')
joblib.dump(scaler,          '../models/ranker_scaler.pkl')
joblib.dump(team_encoder,    '../models/team_encoder.pkl')
joblib.dump(circuit_encoder, '../models/circuit_encoder.pkl')

print("✅ Models saved")
print(f"position_ranker.pkl: {os.path.getsize('../models/position_ranker.pkl')} bytes")


✅ Models saved
position_ranker.pkl: 239847 bytes


In [25]:
import json
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, ndcg_score
from scipy.stats import spearmanr
from xgboost import XGBRegressor

FEATURES_NO_GAP = [
    'grid', 
     'quali_pos', 
     'quali_gap', 
     'driver_form', 
     'circuit_wins_before', 
     'circuit_avg_finish', 
     'constructor_rank_norm', 
     'circuit_type', 
     'team_encoded', 
     'circuit_encoded',
]
FEATURES_WITH_GAP = [
    'grid', 
     'quali_pos', 
     'quali_gap', 
     'driver_form', 
     'circuit_wins_before', 
     'circuit_avg_finish', 
     'constructor_rank_norm', 
     'circuit_type', 
     'team_encoded', 
     'circuit_encoded',

]

def run_full_eval(feature_set, test_year, train_until):
    train_df = df_clean[df_clean['year'] <= train_until]
    test_df  = df_clean[df_clean['year'] == test_year]
    train_df = train_df.dropna(subset=feature_set)
    test_df  = test_df.dropna(subset=feature_set)
    if test_df.empty:
        return None
    X_train = train_df[feature_set]
    y_train = train_df[TARGET]
    X_test  = test_df[feature_set]
    y_test  = test_df[TARGET]

    # Scale for Ridge
    scaler         = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # Ridge baseline
    baseline = Ridge()
    baseline.fit(X_train_scaled, y_train)
    baseline_preds = baseline.predict(X_test_scaled)

    # XGBoost
    xgb_model = XGBRegressor(
        colsample_bytree=0.7, learning_rate=0.05,
        max_depth=3, n_estimators=100,
        subsample=0.9, random_state=42, eval_metric='mae'
    )
    xgb_model.fit(X_train, y_train)
    xgb_preds = xgb_model.predict(X_test)

    # ── Ensemble 1 — optimized for winner accuracy ────────
    best_alpha_winner  = 0.0
    best_winner        = -1

    # ── Ensemble 2 — optimized for Spearman (position) ───
    best_alpha_position = 0.0
    best_spearman       = -999

    for alpha in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        blend = (alpha * baseline_preds) + ((1 - alpha) * xgb_preds)

        # Spearman
        corr, _ = spearmanr(blend, y_test)
        if corr > best_spearman:
            best_spearman        = corr
            best_alpha_position  = alpha

        # Winner accuracy
        check                  = test_df.copy()
        check['predicted_pos'] = blend
        correct                = 0
        total                  = check['round'].nunique()

        for round_num in check['round'].unique():
            race             = check[check['round'] == round_num]
            predicted_winner = race.loc[race['predicted_pos'].idxmin(), 'driver']
            actual_winner    = race.loc[race['won'] == 1, 'driver'].values[0]
            if predicted_winner == actual_winner:
                correct += 1

        winner_acc = correct / total
        if winner_acc > best_winner:
            best_winner       = winner_acc
            best_alpha_winner = alpha

    ensemble_winner_preds   = (best_alpha_winner   * baseline_preds) + ((1 - best_alpha_winner)   * xgb_preds)
    ensemble_position_preds = (best_alpha_position * baseline_preds) + ((1 - best_alpha_position) * xgb_preds)

    def get_metrics(preds, y_test, test_df):
        spearman, _ = spearmanr(preds, y_test)
        mae         = mean_absolute_error(y_test, preds)
        max_pos     = int(y_test.max()) + 1
        ndcg        = ndcg_score(
            (max_pos - y_test.values).reshape(1, -1),
            (max_pos - preds).reshape(1, -1)
        )
        within_3 = (abs(preds - y_test) <= 3).mean() * 100

        test_check                  = test_df.copy()
        test_check['predicted_pos'] = preds
        total                       = test_check['round'].nunique()
        correct                     = 0
        correct_podium              = 0

        for round_num in test_check['round'].unique():
            race             = test_check[test_check['round'] == round_num]
            predicted_winner = race.loc[race['predicted_pos'].idxmin(), 'driver']
            actual_winner    = race.loc[race['won'] == 1, 'driver'].values[0]
            if predicted_winner == actual_winner:
                correct += 1

            predicted_top3 = set(race.nsmallest(3, 'predicted_pos')['driver'])
            actual_top3    = set(race.nsmallest(3, 'position')['driver'])
            if predicted_top3 == actual_top3:
                correct_podium += 1

        return {
            'spearman':   round(float(spearman), 3),
            'ndcg':       round(float(ndcg), 3),
            'mae':        round(float(mae), 2),
            'within_3':   round(float(within_3), 1),
            'winner_acc': round(correct / total * 100, 1),
            'podium_acc': round(correct_podium / total * 100, 1),
            'races':      int(total),
        }

    return {
        'test_year':          test_year,
        'train_until':        train_until,
        'baseline':           get_metrics(baseline_preds,          y_test, test_df),
        'xgboost':            get_metrics(xgb_preds,               y_test, test_df),
        'ensemble_winner':    get_metrics(ensemble_winner_preds,   y_test, test_df),
        'ensemble_position':  get_metrics(ensemble_position_preds, y_test, test_df),
        'best_alpha_winner':  best_alpha_winner,
        'best_alpha_position':best_alpha_position,
    }

# Run all combinations
test_configs = [
    (2016, 2015),
    (2017, 2016),
    (2018, 2017),
    (2019, 2018),
    (2020, 2019),
    (2021, 2020),
    (2022, 2021),
    (2023, 2022),
    (2024, 2023),
    (2025, 2024),
    (2026, 2025),
]

analytics = {
    'without_gap': [],
    'with_gap':    [],
}

for feature_set, key in [(FEATURES_NO_GAP, 'without_gap'), (FEATURES_WITH_GAP, 'with_gap')]:
    print(f"\nRunning {key}...")
    for test_year, train_until in test_configs:
        print(f"  {test_year}...", end=" ")
        result = run_full_eval(feature_set, test_year, train_until)
        if result:
            analytics[key].append(result)
            print(
                f"baseline={result['baseline']['winner_acc']}% "
                f"xgb={result['xgboost']['winner_acc']}% "
                f"ens_winner={result['ensemble_winner']['winner_acc']}% (a={result['best_alpha_winner']}) "
                f"ens_pos={result['ensemble_position']['winner_acc']}% (a={result['best_alpha_position']})"
            )

# Save to JSON
with open('../data/processed/analytics_results.json', 'w') as f:
    json.dump(analytics, f, indent=2)

print("\n✅ Saved to data/processed/analytics_results.json")
print(json.dumps(analytics, indent=2))


Running without_gap...
  2016... baseline=52.4% xgb=47.6% ens_winner=52.4% (a=1.0) ens_pos=47.6% (a=0.9)
  2017... baseline=50.0% xgb=40.0% ens_winner=55.0% (a=0.4) ens_pos=45.0% (a=0.3)
  2018... baseline=42.9% xgb=57.1% ens_winner=57.1% (a=0.0) ens_pos=52.4% (a=0.2)
  2019... baseline=47.6% xgb=33.3% ens_winner=57.1% (a=0.5) ens_pos=38.1% (a=0.2)
  2020... baseline=64.7% xgb=35.3% ens_winner=64.7% (a=0.9) ens_pos=58.8% (a=0.7)
  2021... baseline=59.1% xgb=40.9% ens_winner=59.1% (a=1.0) ens_pos=50.0% (a=0.4)
  2022... baseline=45.5% xgb=27.3% ens_winner=45.5% (a=0.5) ens_pos=40.9% (a=0.9)
  2023... baseline=81.8% xgb=68.2% ens_winner=81.8% (a=0.6) ens_pos=81.8% (a=0.9)
  2024... baseline=33.3% xgb=41.7% ens_winner=41.7% (a=0.0) ens_pos=37.5% (a=0.8)
  2025... baseline=41.7% xgb=45.8% ens_winner=45.8% (a=0.0) ens_pos=41.7% (a=0.2)
  2026... baseline=50.0% xgb=50.0% ens_winner=50.0% (a=0.0) ens_pos=50.0% (a=0.9)

Running with_gap...
  2016... baseline=52.4% xgb=52.4% ens_winner=52.4% (

In [ ]:
# ── 2026 Season Data ──────────────────────────────────────
F1_2026_DRIVERS = [
    {'driver': 'max_verstappen',    'team': 'red_bull'},
    {'driver': 'isack_hadjar',       'team': 'red_bull'},
    {'driver': 'lewis_hamilton',    'team': 'ferrari'},
    {'driver': 'charles_leclerc',   'team': 'ferrari'},
    {'driver': 'kimi_antonelli',    'team': 'mercedes'},
    {'driver': 'george_russell',    'team': 'mercedes'},
    {'driver': 'lando_norris',      'team': 'mclaren'},
    {'driver': 'oscar_piastri',     'team': 'mclaren'},
    {'driver': 'fernando_alonso',   'team': 'aston_martin'},
    {'driver': 'lance_stroll',      'team': 'aston_martin'},
    {'driver': 'pierre_gasly',      'team': 'alpine'},
    {'driver': 'franco_calapinto',       'team': 'alpine'},
    {'driver': 'carlos_sainz',      'team': 'williams'},
    {'driver': 'alexander_albon',   'team': 'williams'},
    {'driver': 'liam_lawson',      'team': 'rb'},
    {'driver': 'arvid_lindblad',    'team': 'rb'},
    {'driver': 'nico_hulkenberg',   'team': 'kick_sauber'},
    {'driver': 'gabriel_bortoleto', 'team': 'kick_sauber'},
    {'driver': 'oliver_bearman',    'team': 'haas'},
    {'driver': 'esteban_ocon',      'team': 'haas'},
    {'driver': 'sergio_perez',      'team':'cadillac'},
    {'driver': 'valtteri_bottas',     'team': 'cadillac'},
]

F1_2026_CIRCUITS = [
    {'round': 1,  'circuit': 'albert_park',   'name': 'Australian GP',        'completed': True},
    {'round': 2,  'circuit': 'shanghai',       'name': 'Chinese GP (Sprint)',  'completed': False},
    {'round': 3,  'circuit': 'suzuka',         'name': 'Japanese GP',          'completed': False},
    {'round': 4,  'circuit': 'bahrain',        'name': 'Bahrain GP',           'completed': False},
    {'round': 5,  'circuit': 'jeddah',         'name': 'Saudi Arabian GP',     'completed': False},
    {'round': 6,  'circuit': 'miami',          'name': 'Miami GP (Sprint)',     'completed': False},
    {'round': 7,  'circuit': 'villeneuve',     'name': 'Canadian GP (Sprint)', 'completed': False},
    {'round': 8,  'circuit': 'monaco',         'name': 'Monaco GP',            'completed': False},
    {'round': 9,  'circuit': 'catalunya',      'name': 'Barcelona GP',         'completed': False},
    {'round': 10, 'circuit': 'red_bull_ring',  'name': 'Austrian GP',          'completed': False},
    {'round': 11, 'circuit': 'silverstone',    'name': 'British GP (Sprint)',   'completed': False},
    {'round': 12, 'circuit': 'spa',            'name': 'Belgian GP',           'completed': False},
    {'round': 13, 'circuit': 'hungaroring',    'name': 'Hungarian GP',         'completed': False},
    {'round': 14, 'circuit': 'zandvoort',      'name': 'Dutch GP (Sprint)',     'completed': False},
    {'round': 15, 'circuit': 'monza',          'name': 'Italian GP',           'completed': False},
    {'round': 16, 'circuit': 'ifema_madrid',   'name': 'Spanish GP',           'completed': False},
    {'round': 17, 'circuit': 'baku',           'name': 'Azerbaijan GP',        'completed': False},
    {'round': 18, 'circuit': 'marina_bay',     'name': 'Singapore GP (Sprint)','completed': False},
    {'round': 19, 'circuit': 'americas',       'name': 'US GP',                'completed': False},
    {'round': 20, 'circuit': 'rodriguez',      'name': 'Mexico City GP',       'completed': False},
    {'round': 21, 'circuit': 'interlagos',     'name': 'São Paulo GP',         'completed': False},
    {'round': 22, 'circuit': 'vegas',          'name': 'Las Vegas GP',         'completed': False},
    {'round': 23, 'circuit': 'losail',         'name': 'Qatar GP',             'completed': False},
    {'round': 24, 'circuit': 'yas_marina',     'name': 'Abu Dhabi GP',         'completed': False},
]
def get_constructor_rank(team):
    """Get most recent constructor rank for this team"""
    team_data = df_clean[df_clean['team'] == team]
    if team_data.empty:
        return 0.5
    latest = team_data.sort_values(['year', 'round']).iloc[-1]
    return float(latest['constructor_rank_norm'])
    
def predict_2026_race(circuit_id, recent_weight=10):
    train_df = df_clean.dropna(subset=FEATURES)

    X_train = train_df[FEATURES]
    y_train = train_df[TARGET]

    # Sample weights — recent seasons weighted more
    weights = np.ones(len(train_df))
    weights[train_df['year'] == 2026] = recent_weight
    weights[train_df['year'] == 2025] = recent_weight * 0.3
    weights[train_df['year'] == 2024] = recent_weight * 0.1

    # Scale for Ridge
    scaler         = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    # Ridge baseline — no sample weights, Ridge doesn't support it cleanly
    baseline = Ridge()
    baseline.fit(X_train_scaled, y_train)

    # XGBoost with sample weights
    xgb_model = XGBRegressor(
        n_estimators     = 300,
        max_depth        = 3,
        learning_rate    = 0.01,
        subsample        = 0.9,
        colsample_bytree = 0.8,
        min_child_weight = 1,
        reg_lambda       = 1,
        random_state     = 42,
        eval_metric      = 'mae'
    )
    xgb_model.fit(X_train, y_train, sample_weight=weights)

    # Build feature rows for all 2026 drivers
    rows = []
    for d in F1_2026_DRIVERS:
        driver = d['driver']
        team   = d['team']

        driver_history = df_clean[df_clean['driver'] == driver]

        if driver_history.empty:
            team_history  = df_clean[df_clean['team'] == team]
            driver_form   = float(team_history['team_win_rate'].mean()) if not team_history.empty else 0.0
            finish_rate   = float(team_history['driver_finish_rate'].mean()) if not team_history.empty else 0.8
            team_win_rate = float(team_history['team_win_rate'].mean()) if not team_history.empty else 0.0
        else:
            latest        = driver_history.sort_values(['year', 'round']).iloc[-1]
            driver_form   = float(latest['driver_form'])
            finish_rate   = float(latest['driver_finish_rate'])
            team_win_rate = float(latest['team_win_rate'])

        circuit_wins = float(df_clean[
            (df_clean['driver']  == driver) &
            (df_clean['circuit'] == circuit_id)
        ]['won'].sum())

        try:    team_enc    = int(team_encoder.transform([team])[0])
        except: team_enc    = 0
        try:    circuit_enc = int(circuit_encoder.transform([circuit_id])[0])
        except: circuit_enc = 0

        rows.append({
        'driver':                driver,
        'team':                  team,
        'grid':                  10,
        'quali_pos':             10,
        'quali_gap':             0.5,
        'grid_penalty':          0,
        'driver_form':           driver_form,
        'team_win_rate':         team_win_rate,
        'circuit_wins_before':   circuit_wins,
        'driver_finish_rate':    finish_rate,
        'constructor_rank_norm': get_constructor_rank(team),
        'starting_compound':     1,   # assume medium — unknown pre-race
        'circuit_type':          CIRCUIT_TYPES.get(circuit_id, 2),
        'team_encoded':          team_enc,
        'circuit_encoded':       circuit_enc,
        'wet_race':              0,     
        'track_temp':            35.0,
        })

    features_df = pd.DataFrame(rows)[FEATURES]
    features_scaled = scaler.transform(features_df)

    # Get predictions from both models
    baseline_preds = baseline.predict(features_scaled)
    xgb_preds      = xgb_model.predict(features_df)

    # Ensemble — use alpha=0.5 as neutral blend since no test year to optimize on
    # Or use the best_alpha from the most recent completed year
    best_alpha     = 0.2
    ensemble_preds = (best_alpha * baseline_preds) + ((1 - best_alpha) * xgb_preds)

    def make_results(preds, model_name):
        inverted  = 1 / np.clip(preds, 0.1, 25)
        win_probs = (inverted / inverted.sum()) * 100
        results   = []
        for i, row in enumerate(rows):
            results.append({
                'model':         model_name,
                'driver':        row['driver'],
                'team':          row['team'],
                'predicted_pos': round(float(preds[i]), 2),
                'win_prob':      round(float(win_probs[i]), 1),
                'driver_form':   round(row['driver_form'], 3),
                'circuit_wins':  int(row['circuit_wins_before']),
            })
        results.sort(key=lambda x: x['predicted_pos'])
        for i, r in enumerate(results):
            r['rank'] = i + 1
        return results

    baseline_results = make_results(baseline_preds, 'Ridge')
    xgb_results      = make_results(xgb_preds,      'XGBoost')
    ensemble_results = make_results(ensemble_preds,  'Ensemble')

    # Print all three side by side
    print(f"\n2026 {circuit_id.upper().replace('_', ' ')} GP — PRE-QUALIFYING PREDICTION")
    print(f"2026 data weighted {recent_weight}x | 2025 weighted {recent_weight * 0.5}x | equal grid assumed")
    print(f"{'─' * 90}")
    print(f"{'POS':<5} {'DRIVER':<22} {'RIDGE':^16} {'XGBOOST':^16} {'ENSEMBLE':^16} {'FORM':<8} {'CKT W'}")
    print(f"{'─' * 90}")

    # Print by ensemble ranking
    for r in ensemble_results:
        driver   = r['driver']
        ridge_r  = next(x for x in baseline_results if x['driver'] == driver)
        xgb_r    = next(x for x in xgb_results      if x['driver'] == driver)

        print(
            f"P{r['rank']:<4} "
            f"{driver:<22} "
            f"P{ridge_r['rank']:<3} {ridge_r['win_prob']:>5.1f}%   "
            f"P{xgb_r['rank']:<3} {xgb_r['win_prob']:>5.1f}%   "
            f"P{r['rank']:<3} {r['win_prob']:>5.1f}%   "
            f"{r['driver_form']:<8.3f}"
            f"{r['circuit_wins']}"
        )

    print(f"\nNo qualifying data — ranking based on historical form + circuit history")

    return {
        'baseline': baseline_results,
        'xgboost':  xgb_results,
        'ensemble': ensemble_results,
    }

# ── Check available circuit IDs first ────────────────────
print("Circuits in your data:")
print(sorted(df_clean['circuit'].unique()))

In [ ]:
# Predict any upcoming race — use exact circuit ID from your data
results = predict_2026_race('bahrain')
results = predict_2026_race('suzuka')
results = predict_2026_race('monaco')

In [26]:
import json
import numpy as np
from itertools import combinations
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr
from xgboost import XGBRegressor

# ── Feature pool ─────────────────────────────────────────────────────────────
# Split into "always include" (core) and "optional" (candidates to search over)

CORE_FEATURES = [
    'grid',
    'quali_pos',
]

OPTIONAL_FEATURES = [
    'quali_gap',
    'grid_penalty',
    'driver_form',
    'team_win_rate',
    'circuit_wins_before',
    'circuit_avg_finish',
    'circuit_overperformance',
    'driver_finish_rate',
    'constructor_rank_norm',
    'constructor_momentum',
    'starting_compound',
    'circuit_type',
    'wet_race',
    'track_temp',
    'team_encoded',
    'circuit_encoded',
]

# ── Search strategy ───────────────────────────────────────────────────────────
# OPTIONS:
#   'exhaustive'  — try ALL 2^N subsets of OPTIONAL_FEATURES (slow for N>15)
#   'min_k_max_k' — try all combos of size MIN_K to MAX_K (more practical)
#   'random'      — sample N_RANDOM random subsets (fast approximation)

SEARCH_STRATEGY = 'min_k_max_k'
MIN_K           = 3     # min optional features to include
MAX_K           = 8     # max optional features to include
N_RANDOM        = 500   # only used if SEARCH_STRATEGY == 'random'

# ── Evaluation years ─────────────────────────────────────────────────────────
# Each (test_year, train_until) pair — winner_acc is averaged across all pairs
TEST_CONFIGS = [
    (2022, 2021),
    (2023, 2022),
    (2024, 2023),
]

TARGET = 'position'  # change if your target column has a different name


# ── Core eval (XGBoost only, optimized for speed) ────────────────────────────
def eval_feature_set(feature_set):
    """Returns average winner_acc across TEST_CONFIGS for the given feature set."""
    winner_accs = []

    for test_year, train_until in TEST_CONFIGS:
        train_df = df_clean[df_clean['year'] <= train_until].dropna(subset=feature_set)
        test_df  = df_clean[df_clean['year'] == test_year].dropna(subset=feature_set)

        if test_df.empty or train_df.empty:
            continue

        X_train, y_train = train_df[feature_set], train_df[TARGET]
        X_test,  y_test  = test_df[feature_set],  test_df[TARGET]

        xgb = XGBRegressor(
            colsample_bytree=0.7, learning_rate=0.05,
            max_depth=3, n_estimators=100,
            subsample=0.9, random_state=42, eval_metric='mae',
            verbosity=0,
        )
        xgb.fit(X_train, y_train)
        preds = xgb.predict(X_test)

        check                  = test_df.copy()
        check['predicted_pos'] = preds
        total, correct         = check['round'].nunique(), 0

        for rnd in check['round'].unique():
            race             = check[check['round'] == rnd]
            predicted_winner = race.loc[race['predicted_pos'].idxmin(), 'driver']
            actual_winner    = race.loc[race['won'] == 1, 'driver'].values[0]
            if predicted_winner == actual_winner:
                correct += 1

        winner_accs.append(correct / total * 100)

    return round(float(np.mean(winner_accs)), 2) if winner_accs else 0.0


# ── Build candidate feature sets ─────────────────────────────────────────────
def build_candidates():
    if SEARCH_STRATEGY == 'exhaustive':
        # All non-empty subsets of OPTIONAL_FEATURES
        candidates = []
        for r in range(1, len(OPTIONAL_FEATURES) + 1):
            for combo in combinations(OPTIONAL_FEATURES, r):
                candidates.append(CORE_FEATURES + list(combo))
        return candidates

    elif SEARCH_STRATEGY == 'min_k_max_k':
        candidates = []
        for r in range(MIN_K, MAX_K + 1):
            for combo in combinations(OPTIONAL_FEATURES, r):
                candidates.append(CORE_FEATURES + list(combo))
        return candidates

    elif SEARCH_STRATEGY == 'random':
        rng        = np.random.default_rng(42)
        seen       = set()
        candidates = []
        while len(candidates) < N_RANDOM:
            k     = rng.integers(MIN_K, MAX_K + 1)
            combo = tuple(sorted(rng.choice(OPTIONAL_FEATURES, k, replace=False)))
            if combo not in seen:
                seen.add(combo)
                candidates.append(CORE_FEATURES + list(combo))
        return candidates

    else:
        raise ValueError(f"Unknown strategy: {SEARCH_STRATEGY}")


# ── Run the search ────────────────────────────────────────────────────────────
if __name__ == '__main__':
    candidates = build_candidates()
    print(f"Strategy: {SEARCH_STRATEGY} → {len(candidates):,} feature sets to evaluate")
    print(f"Test configs: {TEST_CONFIGS}\n")

    results = []

    for i, fset in enumerate(candidates):
        acc = eval_feature_set(fset)
        results.append({'features': fset, 'winner_acc': acc})

        if (i + 1) % 50 == 0 or (i + 1) == len(candidates):
            best_so_far = max(results, key=lambda x: x['winner_acc'])
            print(
                f"[{i+1:>5}/{len(candidates)}]  "
                f"latest={acc:.1f}%  "
                f"best={best_so_far['winner_acc']:.1f}% "
                f"← {best_so_far['features']}"
            )

    # ── Sort and report ───────────────────────────────────────────────────────
    results.sort(key=lambda x: x['winner_acc'], reverse=True)

    print("\n" + "=" * 70)
    print("TOP 20 FEATURE SETS BY WINNER ACCURACY")
    print("=" * 70)
    for rank, r in enumerate(results[:20], 1):
        optional_used = [f for f in r['features'] if f not in CORE_FEATURES]
        print(f"#{rank:>2}  {r['winner_acc']:.1f}%  optional={optional_used}")

    print("\n" + "=" * 70)
    print("BEST FEATURE SET")
    print("=" * 70)
    best = results[0]
    print(f"Winner accuracy : {best['winner_acc']:.1f}%")
    print(f"Full feature set: {best['features']}")

    # ── Feature importance ranking ────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("FEATURE IMPORTANCE (avg winner_acc when feature IS included)")
    print("=" * 70)
    feat_stats = {f: [] for f in OPTIONAL_FEATURES}
    for r in results:
        for f in r['features']:
            if f in feat_stats:
                feat_stats[f].append(r['winner_acc'])

    feat_ranking = sorted(
        [(f, np.mean(v), len(v)) for f, v in feat_stats.items() if v],
        key=lambda x: x[1], reverse=True,
    )
    for feat, avg_acc, count in feat_ranking:
        print(f"  {feat:<30}  avg={avg_acc:.2f}%  (in {count} sets)")

    # ── Save ──────────────────────────────────────────────────────────────────
    output = {
        'strategy':    SEARCH_STRATEGY,
        'test_configs': TEST_CONFIGS,
        'top_20':      results[:20],
        'all_results': results,
        'feature_importance': [
            {'feature': f, 'avg_winner_acc': round(np.mean(v), 2), 'n_sets': len(v)}
            for f, v in feat_stats.items() if v
        ],
    }
    with open('../data/processed/feature_search_results.json', 'w') as fp:
        json.dump(output, fp, indent=2)

    print("\n✅ Saved to data/processed/feature_search_results.json")

Strategy: min_k_max_k → 39,066 feature sets to evaluate
Test configs: [(2022, 2021), (2023, 2022), (2024, 2023)]

[   50/39066]  latest=53.0%  best=58.8% ← ['grid', 'quali_pos', 'quali_gap', 'driver_form', 'team_encoded']
[  100/39066]  latest=48.5%  best=58.8% ← ['grid', 'quali_pos', 'quali_gap', 'driver_form', 'team_encoded']
[  150/39066]  latest=44.2%  best=58.8% ← ['grid', 'quali_pos', 'quali_gap', 'driver_form', 'team_encoded']
[  200/39066]  latest=52.9%  best=58.8% ← ['grid', 'quali_pos', 'quali_gap', 'driver_form', 'team_encoded']
[  250/39066]  latest=50.1%  best=59.0% ← ['grid', 'quali_pos', 'driver_form', 'team_win_rate', 'constructor_momentum']
[  300/39066]  latest=50.1%  best=59.0% ← ['grid', 'quali_pos', 'driver_form', 'team_win_rate', 'constructor_momentum']
[  350/39066]  latest=49.9%  best=59.0% ← ['grid', 'quali_pos', 'driver_form', 'team_win_rate', 'constructor_momentum']
[  400/39066]  latest=44.3%  best=59.0% ← ['grid', 'quali_pos', 'driver_form', 'team_win_rate'